## <u>Setup: Gymnasium</u>

For the RL environments (Cliff Walking here), we use a popular library called **Gymnasium** (the maintained successor to the older `gym` library). Documentation: https://github.com/farama-foundation/gymnasium

Gymnasium is an open source library for developing and comparing RL algorithms. It provides a standard API to communicate between a learning algorithm and an environment, plus a standard set of ready-made environments compliant with that API. Rather than coding an environment from scratch, we create an env object from the library and call standard methods on it.

Browse available environments at: https://gymnasium.farama.org/environments. There are several categories (toy text, Atari, etc). Cliff Walking falls under **toy text**.

In [1]:
!pip install gymnasium


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
!pip install "gymnasium[toy-text]"


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import numpy as np
import gymnasium as gym
import random
import time

---
## <u>Explore the Environment</u>

The env object exposes a handful of standard methods worth knowing:

1. `make()`: create an environment
2. `render()`: show/visualize the environment
3. `reset()`: reset the environment to its starting configuration
4. `close()`: close the environment
5. `step()`: take one action in the environment

`observation_space.n` and `action_space.n` report the *total count* of states and actions respectively (48 states, 4 actions here), not a specific state's index. A specific state's index is computed separately as `row * num_cols + col`, a different thing, worth not conflating with the count itself.

`reset()` returns the starting state plus an extra info dictionary.

Note: in Jupyter, closing a rendered environment window has a known display bug and won't close automatically. Refreshing the page or restarting the kernel is the workaround.

In [4]:
# creating sample environment
env = gym.make("CliffWalking-v1")
print(env)  # checking if we have the environment or not

# checking the environment's parameters
print(env.observation_space.n)  # total number of possible states (48, a 4x12 grid)
print(env.action_space.n)       # total number of possible actions (4: up, down, left, right)

# reset() returns two things: the starting state, and an extra info dict (mostly unused here)
starting_state, extra = env.reset()
print(starting_state)  # should be 36, the encoded index of the agent's fixed starting cell

<OrderEnforcing<PassiveEnvChecker<CliffWalkingEnv<CliffWalking-v1>>>>
48
4
36


---
## <u>SARSA: theory</u>

**SARSA** (State, Action, Reward, State, Action, named after exactly the five values its update rule uses) is an **on-policy** temporal difference control algorithm: it learns Q-values for the exact policy it is currently following, including its exploration behaviour, not for some separate, idealized policy.

**Q-table**: since this environment has a small, finite number of states (48) and actions (4), Q-values can be stored directly in a table: 48 rows by 4 columns, 192 values total, one Q-value per (state, action) pair. `np.zeros((48,4))` initializes all of them to 0, since nothing is known yet.

**Epsilon-greedy policy**: balances exploration (trying new actions to discover their value) against exploitation (taking the currently-best-known action). With probability `epsilon`, take a completely random action (explore). Otherwise, take the action with the highest known Q-value for the current state (exploit). `epsilon = 0.1` means 10% explore, 90% exploit.

**SARSA update rule**:
```
Q(s,a) = Q(s,a) + alpha * [R + gamma * Q(s',a') - Q(s,a)]
```
- `alpha`: the learning rate, how much each new experience updates the existing estimate
- `gamma`: the discount factor, how much future reward is valued relative to immediate reward
- `Q(s',a')`: crucially, this uses the *actual next action the policy will take* (chosen via the same epsilon-greedy policy), not necessarily the best possible next action. This is exactly what makes SARSA on-policy, as opposed to Q-learning (covered separately), which uses the best possible next action regardless of what the policy would actually do.

**Episode ending conditions**: an episode can end two ways.
1. **Termination**: the agent reaches the goal state.
2. **Truncation**: the episode is cut short because it violated some external condition (max steps, max time), not because the task was actually solved.

---
## <u>SARSA Training (no rendering)</u>

Training is normally run without visual rendering, since rendering slows training down considerably. Visual rendering is generally reserved for testing/evaluation, not training. Since this is a first pass at learning the algorithm, both versions are built here (this cell without rendering, the next one with it) specifically to compare training time between them.

In [5]:
# start the timer
start_time = time.perf_counter()

# fixed hyperparameters
gamma = 0.99     # discount factor
alpha = 0.5      # learning rate
epsilon = 0.1    # epsilon for epsilon-greedy: 10% explore, 90% exploit
episodes = 500   # number of full attempts from start state to goal/failure

# Q-table: one Q-value per (state, action) pair
# 48 states * 4 actions = 192 Q-values total
Q_values = np.zeros((48,4))  # initialised to zero, nothing learned yet


def epsilon_greedy(curr_state):
    random_number = random.random()

    if random_number < epsilon:
        return env.action_space.sample()          # explore: any random action from the full action space
    else:
        return np.argmax(Q_values[curr_state])     # exploit: the best known action for this state


# env was already created above; if starting fresh, recreate it first with env = gym.make("CliffWalking-v1")

for episode in range(episodes):

    terminated = False
    truncated = False

    # reset() is called at the START of every episode, not just once overall. Each episode is a fresh, independent attempt, so 
    # the agent needs to be placed back at the starting cell before it begins. Without this, episode 2 would start wherever 
    # episode 1 happened to end (goal or cliff), which defeats the purpose of running many independent episodes. reset() returns 
    # (state, info), that IS how the state is obtained from it, it's simply the function's return value. 
    
    curr_state, extra = env.reset()

    # choose the first action via the epsilon-greedy policy
    curr_action = epsilon_greedy(curr_state)

    total_rewards = 0
    total_steps = 0

    while not (terminated or truncated):

        # step() returns 5 values: next_state, reward, terminated, truncated, and an extra info dict
        next_state, reward, terminated, truncated, extra = env.step(curr_action)

        # choose the next action, also via epsilon-greedy, using the policy currently being followed
        next_action = epsilon_greedy(next_state)

        # SARSA update rule: Q(s,a) = Q(s,a) + alpha * [R + gamma * Q(s',a') - Q(s,a)]
        td_target = reward + gamma * Q_values[next_state, next_action]
        td_error = td_target - Q_values[curr_state, curr_action]
        Q_values[curr_state, curr_action] = Q_values[curr_state, curr_action] + alpha * td_error

        # move forward: next becomes current, for both state and action
        curr_state = next_state
        curr_action = next_action

        total_rewards = total_rewards + reward
        total_steps = total_steps + 1

    # print progress every 50th episode
    if episode % 50 == 0:
        print(f"for episode {episode+1}/{episodes} --------> total rewards = {total_rewards} -------> total steps = {total_steps}")

env.close()

end_time = time.perf_counter()
print(f"Non-Visual Training finished in {end_time - start_time:.2f} seconds.")

for episode 1/500 --------> total rewards = -130 -------> total steps = 130
for episode 51/500 --------> total rewards = -23 -------> total steps = 23
for episode 101/500 --------> total rewards = -260 -------> total steps = 260
for episode 151/500 --------> total rewards = -53 -------> total steps = 53
for episode 201/500 --------> total rewards = -20 -------> total steps = 20
for episode 251/500 --------> total rewards = -31 -------> total steps = 31
for episode 301/500 --------> total rewards = -23 -------> total steps = 23
for episode 351/500 --------> total rewards = -19 -------> total steps = 19
for episode 401/500 --------> total rewards = -21 -------> total steps = 21
for episode 451/500 --------> total rewards = -119 -------> total steps = 20
Non-Visual Training finished in 0.15 seconds.


---
## <u>SARSA Training (with rendering)</u>

Not advisable for future projects (rendering meaningfully slows training), but useful here to see it happening and to compare timing directly against the non-visual run above.

In [6]:
# re-initialise the Q-table fresh, it's already filled in from the training run above,
# and reusing it here would make this an unfair, head-start comparison rather than a clean one
Q_values = np.zeros((48,4))

start_time = time.perf_counter()

gamma = 0.99
alpha = 0.5
epsilon = 0.1
episodes = 500

for episode in range(episodes):

    # same loop as before, but every 50th episode gets a renderable (visual) environment,
    # every other episode uses a normal, non-rendered one
    if episode % 50 == 0:
        env = gym.make("CliffWalking-v1", render_mode = "human")
    else:
        env = gym.make("CliffWalking-v1")

    curr_state, extra = env.reset()
    curr_action = epsilon_greedy(curr_state)

    terminated = False
    truncated = False

    total_rewards = 0
    total_steps = 0

    while not (terminated or truncated):

        next_state, reward, terminated, truncated, extra = env.step(curr_action)
        next_action = epsilon_greedy(next_state)

        # SARSA update, identical to the non-visual version
        td_target = reward + gamma * Q_values[next_state, next_action]
        td_error = td_target - Q_values[curr_state, curr_action]
        Q_values[curr_state, curr_action] = Q_values[curr_state, curr_action] + alpha * td_error

        curr_state = next_state
        curr_action = next_action

        total_rewards = total_rewards + reward
        total_steps = total_steps + 1

    if episode % 50 == 0:
        print(f"for episode {episode+50}/{episodes} --------> total rewards = {total_rewards} -------> total steps = {total_steps}")

env.close()  

end_time = time.perf_counter()
print(f"Visual Training finished in {end_time - start_time:.2f} seconds")  

for episode 50/500 --------> total rewards = -95 -------> total steps = 95
for episode 100/500 --------> total rewards = -18 -------> total steps = 18
for episode 150/500 --------> total rewards = -20 -------> total steps = 20
for episode 200/500 --------> total rewards = -20 -------> total steps = 20
for episode 250/500 --------> total rewards = -18 -------> total steps = 18
for episode 300/500 --------> total rewards = -19 -------> total steps = 19
for episode 350/500 --------> total rewards = -22 -------> total steps = 22
for episode 400/500 --------> total rewards = -22 -------> total steps = 22
for episode 450/500 --------> total rewards = -19 -------> total steps = 19
for episode 500/500 --------> total rewards = -18 -------> total steps = 18
Visual Training finished in 72.38 seconds


In [7]:
# actual testing to see what our agent learned

# this is time we will used the learned Q-values

# make the environment render friendly
env = gym.make("CliffWalking-v1", render_mode = "human")

# get the starting state as current state
curr_state, extra = env.reset()

# initialise hyperparameters and the flag values for the testing loop
terminated = False
truncated = False
total_rewards = 0
total_steps = 0

while not (terminated or truncated):

    # choose the current action, this time rather than asking the policy, we will just pick the optimal action usnig Q-values
    curr_action = np.argmax(Q_values[curr_state])

    # take curr_action
    next_state, reward, terminated, truncated, extra = env.step(curr_action)

    # update rewards and steps
    total_rewards = total_rewards + reward
    total_steps = total_steps + 1

    # updating the current state
    # no updating actions as we arent using policy anymore
    curr_state = next_state

env.close()
print(f"total reward = {total_rewards} & episode len = {total_steps}")

total reward = -17 & episode len = 17
